# 06 - Regras temporais e guardrails

Compara hipoteses pre-registradas em calibracao e em cinco dias internos posteriores. Nenhuma regra e promovida automaticamente.

In [ ]:
from pathlib import Path
import json, os, sys
import pandas as pd
ROOT = Path.cwd(); NB_DIR = ROOT / 'notebooks' if (ROOT / 'notebooks').is_dir() else ROOT
sys.path.insert(0, str(NB_DIR))
from produtividade_30d import carregar_fontes, preparar_dataset, dividir_por_dia, aplicar_regras, avaliar_politicas
OUT = Path(os.environ.get('KV_30D_OUTPUT_DIR', NB_DIR / 'outputs' / 'productivity_30d')); OUT.mkdir(parents=True, exist_ok=True)
eventos, catalogo, _ = carregar_fontes(); proxy, _ = preparar_dataset(eventos, catalogo); split = dividir_por_dia(proxy)
resultados = []
for parte, df in [('calibracao', split.calibracao), ('teste_interno', split.teste_interno)]:
    tabela = avaliar_politicas(df, aplicar_regras(df)); tabela.insert(0, 'parte', parte); resultados.append(tabela)
resultado = pd.concat(resultados, ignore_index=True)
gate = {'precision_I_min': 0.80, 'recall_I_min': 0.70, 'precision_P_queda_max_pp': 2, 'coverage_min': 0.60, 'false_accusation_max': 0.05, 'claims_I_min_holdout': 20}
resultado['passa_screening'] = (resultado.precision_I >= gate['precision_I_min']) & (resultado.recall_I >= gate['recall_I_min']) & (resultado.coverage >= gate['coverage_min']) & (resultado.false_accusation_rate <= gate['false_accusation_max']) & (resultado.claims_I >= 10)
resultado.to_csv(OUT / 'regras_temporais_screening.csv', index=False)
(OUT / 'gate_promocao.json').write_text(json.dumps(gate, indent=2), encoding='utf-8')
display(resultado)
print('Regra so pode seguir para holdout se passar calibracao E teste interno; este notebook nao altera producao.')